# FactCheck-RAG — Colab runner

Runs the scripts of the repository on a Colab GPU.

* **Code** is cloned fresh from GitHub into `/content/FactCheck_RAG` each session (always the latest commit).
* **Index and results** live on Google Drive (`MyDrive/factcheck_rag/data` and `.../outputs`) and are linked into the repo, so they survive the end of the session.

**Runtime:** *Runtime → Change runtime type → T4 GPU* (CPU is enough for step 3 with `--no-embed`).
Teammates: add a shortcut to the shared `factcheck_rag` Drive folder in *My Drive* first.

## 1. Drive + paths

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/factcheck_rag'   # index + results (persistent)
REPO_DIR  = '/content/FactCheck_RAG'                  # code (re-cloned each session)
REPO_URL  = 'https://github.com/sihamErmk/FactCheck_RAG.git'
BRANCH    = 'main'

# To test a branch that is not merged yet (e.g. from a fork), uncomment:
# REPO_URL = 'https://github.com/aya2ait/FactCheck_RAG.git'
# BRANCH   = 'phase3-repair'

## 2. Get the code

If the repository is private, add a GitHub token in Colab *Secrets* (key icon on the left) named `GITHUB_TOKEN`, with notebook access enabled.

In [ ]:
import os, subprocess
url = REPO_URL
try:
    from google.colab import userdata
    token = userdata.get('GITHUB_TOKEN')
    url = REPO_URL.replace('https://', f'https://{token}@')
except Exception:
    pass  # public repo, no token needed

if os.path.isdir(REPO_DIR):
    subprocess.run(['git', '-C', REPO_DIR, 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', '-q', '-b', BRANCH, url, REPO_DIR], check=True)
%cd {REPO_DIR}
!git log --oneline -3

## 3. Link data/ and outputs/ to Drive, install dependencies

In [ ]:
for name in ['data', 'outputs']:
    target = os.path.join(DRIVE_DIR, name)
    os.makedirs(target, exist_ok=True)
    link = os.path.join(REPO_DIR, name)
    if not os.path.islink(link):
        os.symlink(target, link)
!ls -lh data/ outputs/
!pip install -q -r requirements.txt
!python tests/test_text_parsing.py

## 4. Build the index (only once)

Skipped automatically when `data/faiss.index` already exists. On a CPU runtime use `--no-embed` (download only, ~10 min), then switch to GPU for the embedding (~15 min).

In [ ]:
if os.path.exists('data/faiss.index'):
    print('Index already built:', os.path.getsize('data/faiss.index') // 2**20, 'MB')
else:
    !python scripts/build_index.py

## 5. Baseline RAG (Phase 1)

`--limit 5` for a quick test; remove it for the 200 questions. Runs resume where they stopped.

In [ ]:
!python scripts/run_baseline.py --limit 5

## 6. Claim verification (Phase 2)

Delete `outputs/verified_*.jsonl` first if you changed the decomposer or a verifier.

In [ ]:
# !rm -f outputs/verified_nli.jsonl outputs/verified_judge.jsonl
!python scripts/run_verify.py --verifier nli --limit 5
!python scripts/run_verify.py --verifier judge --limit 5

## 7. Inspect: NLI vs LLM judge, claim by claim

In [ ]:
import json
nli = [json.loads(l) for l in open('outputs/verified_nli.jsonl')]
jdg = [json.loads(l) for l in open('outputs/verified_judge.jsonl')]
for a, b in zip(nli, jdg):
    print('=' * 80, '\nQ :', a['question'])
    print('Titles :', [p['title'] for p in a['passages']])
    print('Answer :', a['answer'])
    for ca, cb in zip(a['check']['claims'], b['check']['claims']):
        flag = '  ' if ca['supported'] == cb['supported'] else '≠ '
        print(f"{flag}NLI={ca['supported']!s:5} ({ca['score']:.2f})  JUDGE={cb['supported']!s:5} {ca['citations']} | {ca['text']}")
        print(f"      quote: {cb.get('quote', '')[:120]}")

## 8. Repair loop (Phase 3)

Repairs the baseline drafts: for every unsupported claim, a targeted sub-query retrieves new passages and only that sentence is rewritten, then re-verified (at most `--max-rounds` rounds). Delete `outputs/factcheck_*.jsonl` first if you changed the code.

In [ ]:
# !rm -f outputs/factcheck_nli.jsonl outputs/factcheck_judge.jsonl
!python scripts/run_factcheck.py --verifier nli --limit 5

## 9. Inspect the repairs: draft vs final answer, sub-queries and rewrites

In [ ]:
import json
for l in open('outputs/factcheck_nli.jsonl'):
    r = json.loads(l)
    print('=' * 80, '\nQ :', r['question'])
    print('Claim P per round:', [s['claim_precision'] for s in r['snapshots']], '| rounds used:', r['n_rounds'])
    if r['n_rounds'] == 0:
        print('(nothing to repair)')
        continue
    print('DRAFT :', r['draft_answer'])
    for rnd in r['rounds']:
        print(f"-- round {rnd['round']}")
        for q in rnd['subqueries']:
            print(f"   claim : {q['claim']}\n   query : {q['query']}  -> passages {q['passages']}")
        for w in rnd['rewrites']:
            print(f"   BEFORE: {w['before']}\n   AFTER : {w['after']}")
    print('FINAL :', r['answer'])
    for c in r['claims']:
        if not c['supported']:
            print('   still unsupported:', c['text'])

## 10. Before closing

Forces Drive to save everything, then *Runtime → Disconnect and delete runtime* to stop using GPU quota.

In [ ]:
drive.flush_and_unmount()